In [1]:
# Cell 2: Imports and settings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

# Filepaths (assume notebook cwd is repo folder)
ECHONEST = 'echonest_features.tsv'
SPECTRAL = 'spectral_features.tsv'
TRACKS = 'tracks.tsv'
GENRES = 'genres.csv'

In [2]:
# Cell 3: Load TSVs (low_memory=False to avoid mixed types)
echonest = pd.read_csv(ECHONEST, sep='	', low_memory=False)
spectral = pd.read_csv(SPECTRAL, sep='	', low_memory=False)
tracks = pd.read_csv(TRACKS, sep='	', low_memory=False)
# genres.csv may be comma-separated; guard for both separators
try:
    genres_df = pd.read_csv(GENRES)
except Exception:
    genres_df = pd.read_csv(GENRES, sep=';')

print('echonest', echonest.shape)
print('spectral', spectral.shape)
print('tracks', tracks.shape)
print('genres', genres_df.shape)

echonest (11440, 9)
spectral (99995, 22)
tracks (99995, 14)
genres (164, 5)


In [3]:
# Cell 4: Clean column names (lowercase, replace spaces with underscore)
def clean_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    return df

echonest = clean_cols(echonest)
spectral = clean_cols(spectral)
tracks = clean_cols(tracks)
genres_df = clean_cols(genres_df)

# Show columns sample
print('echonest cols:', echonest.columns.tolist())
print('spectral cols sample:', spectral.columns.tolist()[:20])
print('tracks cols:', tracks.columns.tolist())

echonest cols: ['track_id', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'speechiness', 'tempo', 'valence']
spectral cols sample: ['track_id', 'spectral_bandwidth_kurtosis_01', 'spectral_bandwidth_max_01', 'spectral_bandwidth_mean_01', 'spectral_bandwidth_median_01', 'spectral_bandwidth_min_01', 'spectral_bandwidth_skew_01', 'spectral_bandwidth_std_01', 'spectral_centroid_kurtosis_01', 'spectral_centroid_max_01', 'spectral_centroid_mean_01', 'spectral_centroid_median_01', 'spectral_centroid_min_01', 'spectral_centroid_skew_01', 'spectral_centroid_std_01', 'spectral_rolloff_kurtosis_01', 'spectral_rolloff_max_01', 'spectral_rolloff_mean_01', 'spectral_rolloff_median_01', 'spectral_rolloff_min_01']
tracks cols: ['track_id', 'album_title', 'album_tracks', 'artist_latitude', 'artist_longitude', 'artist_name', 'duration', 'favorites', 'genre_top', 'genres', 'genres_all', 'interest', 'listens', 'title']


In [4]:
# Cell 5: Merge datasets with an outer join on track_id to preserve all tracks
merged = tracks.merge(echonest, on='track_id', how='outer')
merged = merged.merge(spectral, on='track_id', how='outer')
print('Merged shape:', merged.shape)

# Quick look at head and missing fraction
display(merged.head(3))
miss_frac = merged.isna().mean().sort_values(ascending=False)
print('Top 20 missing fractions:', miss_frac.head(20))

Merged shape: (100882, 43)


,track_id,album_title,album_tracks,artist_latitude,artist_longitude,artist_name,duration,favorites,genre_top,genres,...,spectral_centroid_min_01,spectral_centroid_skew_01,spectral_centroid_std_01,spectral_rolloff_kurtosis_01,spectral_rolloff_max_01,spectral_rolloff_mean_01,spectral_rolloff_median_01,spectral_rolloff_min_01,spectral_rolloff_skew_01,spectral_rolloff_std_01
0,11870,Wildahead Portibeast,10.0,NaN,NaN,Wildahead Portibeast,131.0,0.0,Hip-Hop,[21],...,327.569489,2.080017,575.112488,2.839292,8968.579102,2338.119629,1981.054688,516.796875,1.812383,1234.268433
1,11871,Wildahead Portibeast,10.0,NaN,NaN,Wildahead Portibeast,185.0,0.0,Hip-Hop,[21],...,187.277390,1.886271,655.114319,2.036670,9560.742188,2132.796143,1830.322266,226.098633,1.412759,1387.095459
2,11872,Wildahead Portibeast,10.0,NaN,NaN,Wildahead Portibeast,183.0,0.0,Hip-Hop,[21],...,99.604340,1.492531,645.915894,-0.463741,9345.410156,2760.100342,2583.984375,64.599609,0.245426,1420.888672


Top 20 missing fractions: speechiness                   0.888087
valence                       0.886808
danceability                  0.886789
instrumentalness              0.886600
energy                        0.886600
tempo                         0.886600
liveness                      0.886600
acousticness                  0.886600
artist_latitude               0.601445
artist_longitude              0.601445
genre_top                     0.557572
album_title                   0.017823
title                         0.008802
favorites                     0.008792
genres_all                    0.008792
artist_name                   0.008792
duration                      0.008792
album_tracks                  0.008792
genres                        0.008792
spectral_bandwidth_skew_01    0.008792
dtype: float64


## Data quality checks
- Count duplicates on `track_id`
- Inspect `genre_top` coverage (target for Task 1)
- Short analysis of numeric distributions and extreme outliers

In [5]:
# Cell 6: Duplicates and genre_top coverage
dups = merged['track_id'].duplicated().sum()
print('Duplicate track_id rows:', dups)

# Normalize blank strings to NaN for genre_top
merged['genre_top'] = merged['genre_top'].replace('', pd.NA)
print('genre_top missing:', merged['genre_top'].isna().sum(), 'of', len(merged))
print('Top genre counts (sample):')
print(merged['genre_top'].value_counts(dropna=False).head(20))

Duplicate track_id rows: 0
genre_top missing: 56249 of 100882
Top genre counts (sample):
genre_top
NaN                    56249
Rock                   12085
Experimental            9752
Electronic              8769
Hip-Hop                 3334
Folk                    2375
Pop                     2235
Instrumental            2003
International           1256
Classical               1129
Old-Time / Historic      491
Jazz                     484
Spoken                   323
Country                  163
Soul-RnB                 131
Blues                     79
Easy Listening            24
Name: count, dtype: int64


## Inference for missing `genre_top` from `genres_all` using `genres.csv` mapping (optional)
If `genre_top` is missing, we try to map the first id in `genres_all` to a name using `genres.csv`. This helps increase labeled coverage for Task 1.

In [6]:
# Cell 7: Build genre id -> name mapping from genres_df if columns exist
genre_map = {}
if 'genre_id' in genres_df.columns and 'genre_top' in genres_df.columns:
    try:
        genre_map = dict(zip(genres_df['genre_id'].astype(int), genres_df['genre_top']))
    except Exception:
        # fallback: try other possible column names
        if 'id' in genres_df.columns and 'name' in genres_df.columns:
            genre_map = dict(zip(genres_df['id'].astype(int), genres_df['name']))

def infer_genre_from_list(s):
    if pd.isna(s) or str(s).strip() in ('', '[]'):
        return pd.NA
    vals = [t.strip() for t in str(s).strip('[]').split(',') if t.strip()]
    if not vals:
        return pd.NA
    try:
        gid = int(vals[0])
        return genre_map.get(gid, pd.NA)
    except Exception:
        return pd.NA

merged['inferred_genre'] = merged.get('genres_all', pd.Series([pd.NA]*len(merged))).apply(lambda s: infer_genre_from_list(s))
# Fill genre_top where missing with inferred genre
merged['genre_top'] = merged['genre_top'].fillna(merged['inferred_genre'])
merged['genre_top'] = merged['genre_top'].fillna('unknown')
print('After inference, genre_top missing:', merged['genre_top'].isna().sum())
print('Unique genres (sample):', merged['genre_top'].nunique())

After inference, genre_top missing: 0
Unique genres (sample): 17


## Coarse-grained genre mapping (Task 2)
Define a mapping from fine-grained `genre_top` to 4 coarse categories. Edit the mapping to suit your choices and dataset distribution. This notebook uses a suggested grouping to get started.

In [7]:
# Cell 8: Define coarse mapping — adjust as needed
# This is a simple example grouping; refine using actual counts and domain knowledge
rock_pop = set(['rock','pop','indie','alternative'])
hiphop_electronic = set(['hip-hop','electronic','dance','edm','hip hop'])
folk_world = set(['folk','world','international','classical','jazz'])

def coarse_genre(g):
    if pd.isna(g) or g == 'unknown':
        return 'other'
    gg = str(g).lower()
    if any(k in gg for k in ['hip-hop','hip hop','rap']):
        return 'hiphop_electronic'
    if any(k in gg for k in ['electronic','dance','edm']):
        return 'hiphop_electronic'
    if any(k in gg for k in ['rock','pop','indie','alternative']):
        return 'rock_pop'
    if any(k in gg for k in ['folk','world','international','classical','jazz','blues']):
        return 'folk_world'
    return 'other'

merged['genre_coarse'] = merged['genre_top'].apply(coarse_genre)
print('Coarse genre counts:')
print(merged['genre_coarse'].value_counts())

Coarse genre counts:
genre_coarse
other                69136
rock_pop             14320
hiphop_electronic    12103
folk_world            5323
Name: count, dtype: int64


## Feature selection and preprocessing

In [8]:
# Cell 9: Select useful features
# Start with: echonest high-level features + a small set of spectral summaries + metadata like listens, interest
echonest_features = ['acousticness','danceability','energy','instrumentalness','liveness','speechiness','tempo','valence']
spectral_sample = [c for c in merged.columns if 'spectral_centroid_mean' in c or 'spectral_rolloff_mean' in c or 'spectral_bandwidth_mean' in c]
meta_features = ['duration','interest','listens','favorites']
# Keep only features that exist in merged
features = [f for f in echonest_features + spectral_sample + meta_features if f in merged.columns]
print('Using features sample (count):', len(features))
print(features)

X = merged[features].copy()
# Target columns for tasks
y_task1 = merged['genre_top'].copy()  # original fine-grained
y_task2 = merged['genre_coarse'].copy()  # coarse-grained
y_task3 = merged['duration'].copy()  # regression target

Using features sample (count): 15
['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'speechiness', 'tempo', 'valence', 'spectral_bandwidth_mean_01', 'spectral_centroid_mean_01', 'spectral_rolloff_mean_01', 'duration', 'interest', 'listens', 'favorites']


In [9]:
# Cell 10: Impute numeric features (median) and scale; prepare function to build pipelines
numeric_features = X.columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
], remainder='drop')

# Apply preprocessing to get an array for modeling (safe for small to medium datasets)
X_proc = preprocessor.fit_transform(X)
print('Preprocessed feature matrix shape:', X_proc.shape)

Preprocessed feature matrix shape: (100882, 15)


## Modeling utilities: evaluation and training helper functions

In [10]:
# Cell 11: Helpers to evaluate classifiers and regressors
from sklearn.model_selection import cross_validate

def evaluate_classifier(model, X, y, cv=5):
    scoring = ['accuracy','f1_macro']
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    return {k: np.mean(v) for k,v in scores.items()}

def evaluate_regressor(model, X, y, cv=5):
    scoring = {'neg_rmse': 'neg_root_mean_squared_error','r2': 'r2','neg_mae':'neg_mean_absolute_error'}
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    # convert neg metrics
    return {k: np.mean(v) for k,v in scores.items()}

## Task 1 — Predict original genre (`genre_top`)
Use rows where `genre_top` is not `'unknown'` (or adjust if you prefer to keep unknown). We evaluate several classifiers with cross-validation and show a baseline confusion report on a hold-out test set.

In [11]:
# Cell 12: Prepare data for Task 1 (drop unknown)
mask1 = (y_task1.notna()) & (y_task1 != 'unknown')
X1 = X_proc[mask1.values]
y1 = y_task1[mask1].values
print('Task1 samples:', X1.shape[0])

# If too many classes with low support, consider keeping only top-K frequent classes
from collections import Counter
cnt = Counter(y1)
print('Top genres sample:', cnt.most_common(10))

Task1 samples: 44633
Top genres sample: [('Rock', 12085), ('Experimental', 9752), ('Electronic', 8769), ('Hip-Hop', 3334), ('Folk', 2375), ('Pop', 2235), ('Instrumental', 2003), ('International', 1256), ('Classical', 1129), ('Old-Time / Historic', 491)]


In [12]:
# Cell 13: Small set of classifiers to compare
models_clf = [
    ('logreg', LogisticRegression(max_iter=1000)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('svc', SVC()),
    ('dt', DecisionTreeClassifier()),
    ('rf', RandomForestClassifier(n_estimators=100, n_jobs=-1)),
    ('gb', GradientBoostingClassifier()),
    ('mlp', MLPClassifier(max_iter=500))
]
results_task1 = {}
for name, model in models_clf:
    try:
        res = evaluate_classifier(model, X1, y1, cv=4)
        results_task1[name] = res
        print(name, res)
    except Exception as e:
        print('Error for', name, e)

logreg {'fit_time': np.float64(40.744264245033264), 'score_time': np.float64(0.12134784460067749), 'test_accuracy': np.float64(0.42609667153532244), 'test_f1_macro': np.float64(0.1613461878641732)}
knn {'fit_time': np.float64(0.24846899509429932), 'score_time': np.float64(1.8516259789466858), 'test_accuracy': np.float64(0.38552200162486994), 'test_f1_macro': np.float64(0.1644293120374222)}
knn {'fit_time': np.float64(0.24846899509429932), 'score_time': np.float64(1.8516259789466858), 'test_accuracy': np.float64(0.38552200162486994), 'test_f1_macro': np.float64(0.1644293120374222)}
svc {'fit_time': np.float64(170.3137083053589), 'score_time': np.float64(94.87658494710922), 'test_accuracy': np.float64(0.4187933665607273), 'test_f1_macro': np.float64(0.11506314015525002)}
svc {'fit_time': np.float64(170.3137083053589), 'score_time': np.float64(94.87658494710922), 'test_accuracy': np.float64(0.4187933665607273), 'test_f1_macro': np.float64(0.11506314015525002)}
dt {'fit_time': np.float64(2

KeyboardInterrupt: 

### Hold-out test example for a best model (pick a model from results)

In [ ]:
# Cell 14: create a stratified train/test split and evaluate one selected model (e.g., RandomForest)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1 macro:', f1_score(y_test, y_pred, average='macro'))
print('Classification report:')
print(classification_report(y_test, y_pred))

## Task 2 — Coarse-grained genre (3–4 categories)
This task typically generalizes better. We'll use `genre_coarse` as defined earlier. We'll encode categories and evaluate the same set of classifiers.

In [ ]:
# Cell 15: Prepare Task 2 data - drop 'other' if desired or keep it
mask2 = merged['genre_coarse'].notna()
X2 = X_proc[mask2.values]
y2 = merged.loc[mask2, 'genre_coarse'].values
print('Task2 samples:', X2.shape[0])
print('Class distribution:', pd.Series(y2).value_counts())

# Evaluate same classifiers on coarse labels
results_task2 = {}
for name, model in models_clf:
    try:
        res = evaluate_classifier(model, X2, y2, cv=4)
        results_task2[name] = res
        print(name, res)
    except Exception as e:
        print('Error for', name, e)

## Task 3 — Predict track duration (regression)
We'll predict `duration` using the same features. We'll evaluate several regressors and use RMSE/MAE/R2 metrics.

In [ ]:
# Cell 16: Prepare Task 3 data (drop NaN durations)
mask3 = ~y_task3.isna()
X3 = X_proc[mask3.values]
y3 = y_task3[mask3].values
print('Task3 samples:', X3.shape[0])

models_reg = [
    ('lr', LinearRegression()),
    ('ridge', Ridge()),
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('dt', DecisionTreeRegressor()),
    ('rf', RandomForestRegressor(n_estimators=100, n_jobs=-1)),
    ('gb', GradientBoostingRegressor()),
    ('mlp', MLPRegressor(max_iter=500))
]
results_task3 = {}
for name, model in models_reg:
    try:
        res = evaluate_regressor(model, X3, y3, cv=4)
        results_task3[name] = res
        print(name, res)
    except Exception as e:
        print('Error for', name, e)

## Save preprocessed dataset and results
We save a TSV with the selected features and the three targets for reproducibility and further experiments.

In [ ]:
# Cell 17: Build final preprocessed DataFrame and save
# Convert processed features back to dataframe
feat_names = numeric_features
X_proc_df = pd.DataFrame(X_proc, columns=feat_names, index=merged.index)
final_df = pd.concat([merged[['track_id','genre_top','genre_coarse','duration','interest','listens']], X_proc_df], axis=1)
OUT = 'challenge2_merged_preprocessed.tsv'
final_df.to_csv(OUT, sep='	', index=False)
print('Saved preprocessed dataset to', OUT)

## Next steps and recommendations
- Hyperparameter tuning (GridSearchCV) for the best models found above.
- If class imbalance hurts performance, try class weighting, oversampling (SMOTE), or under-sampling.
- For spectral features: explore PCA to reduce dimensionality or use specialized audio models (CNNs on spectrograms) for improved performance.
- When using XGBoost/LightGBM, install the packages and re-run the boosting experiments for likely better performance.
- Evaluate with stratified splits over time or artist to detect leakage if data contains multiple tracks by the same artist.

### Run instructions (PowerShell)
In the repository folder run:
```powershell
# Launch Jupyter and open this notebook
jupyter notebook

# Or execute end-to-end (produces outputs in-place)
jupyter nbconvert --execute challenge2_pipeline.ipynb --to notebook --inplace
```
If you want, I can: add GridSearch cells, add SMOTE sampling, or integrate XGBoost/LightGBM cells. Which should I add next?